In [1]:
!pip install geopandas folium

In [4]:
import pandas as pd

data_laporan = {
    "id_laporan": [
        "AST-001", "AST-002", "AST-003", "AST-004",
        "AST-005", "AST-006", "AST-007", "AST-008"
    ],
    "nama_aset": [
        "Pohon Pelindung Jl. WR Supratman (Dekat Gerbang UNIB)",
        "Area Rumput (Berm) Simpang Kandang Limun",
        "Pohon Tua Depan Dekanat FT UNIB",
        "Lampu & Bangku Taman Danau Dendam Tak Sudah",
        "Pohon Rawan Tumbang Jl. Kalimantan (Rawa Makmur)",
        "Taman Bermain Publik Pantai Panjang",
        "Area Rumput (Berm) Rumah Lansia Jl. Indragiri",
        "Pohon Miring Dekat RSUD M. Yunus"
    ],
    "kategori": [
        "Arborist / Pohon", "Berm / Rumput", "Arborist / Pohon", "Fasilitas Taman",
        "Arborist / Pohon", "Fasilitas Taman", "Berm / Rumput", "Arborist / Pohon"
    ],
    "kondisi_dilaporkan": [
        "Batang miring ke jalan raya & rawan tumbang",
        "Rumput sangat tinggi menghalangi trotoar",
        "Dahan besar lapuk di atas area parkir",
        "Lampu taman mati & kursi rusak",
        "Akar terangkat akibat angin kencang, darurat",
        "Cat mengelupas & perlu perawatan rutin",
        "Warga lansia melapor tidak sanggup memangkas rumput",
        "Cabang pohon menyentuh kabel jaringan"
    ],
    "tingkat_kerusakan_skor": [90, 45, 85, 65, 95, 40, 50, 88],
    "latitude": [-3.7592, -3.7615, -3.7578, -3.7950, -3.7640, -3.8120, -3.8015, -3.8285],
    "longitude": [102.2735, 102.2760, 102.2710, 102.2980, 102.2810, 102.2610, 102.2690, 102.3160]
}


df = pd.DataFrame(data_laporan)

df.to_csv("data_laporan_aset_kota.csv", index=False)

print("File 'data_laporan_aset_kota.csv' berhasil dibuat")

df

File 'data_laporan_aset_kota.csv' berhasil dibuat


,id_laporan,nama_aset,kategori,kondisi_dilaporkan,tingkat_kerusakan_skor,latitude,longitude
0,AST-001,Pohon Pelindung Jl. WR Supratman (Dekat Gerban...,Arborist / Pohon,Batang miring ke jalan raya & rawan tumbang,90,-3.7592,102.2735
1,AST-002,Area Rumput (Berm) Simpang Kandang Limun,Berm / Rumput,Rumput sangat tinggi menghalangi trotoar,45,-3.7615,102.2760
2,AST-003,Pohon Tua Depan Dekanat FT UNIB,Arborist / Pohon,Dahan besar lapuk di atas area parkir,85,-3.7578,102.2710
3,AST-004,Lampu & Bangku Taman Danau Dendam Tak Sudah,Fasilitas Taman,Lampu taman mati & kursi rusak,65,-3.7950,102.2980
4,AST-005,Pohon Rawan Tumbang Jl. Kalimantan (Rawa Makmur),Arborist / Pohon,"Akar terangkat akibat angin kencang, darurat",95,-3.7640,102.2810
5,AST-006,Taman Bermain Publik Pantai Panjang,Fasilitas Taman,Cat mengelupas & perlu perawatan rutin,40,-3.8120,102.2610
6,AST-007,Area Rumput (Berm) Rumah Lansia Jl. Indragiri,Berm / Rumput,Warga lansia melapor tidak sanggup memangkas r...,50,-3.8015,102.2690
7,AST-008,Pohon Miring Dekat RSUD M. Yunus,Arborist / Pohon,Cabang pohon menyentuh kabel jaringan,88,-3.8285,102.3160


In [5]:
import geopandas as gpd
from shapely.geometry import Point

geometri_titik = [Point(xy) for xy in zip(df["longitude"], df["latitude"])]

gdf = gpd.GeoDataFrame(df, geometry=geometri_titik, crs="EPSG:4326")

gdf_utm = gdf.to_crs(epsg=32748)

titik_posko_gps = gpd.GeoSeries([Point(102.2725, -3.7585)], crs="EPSG:4326")
titik_posko_utm = titik_posko_gps.to_crs(epsg=32748).iloc[0]

gdf["jarak_posko_m"] = gdf_utm.geometry.distance(titik_posko_utm).round(1)

def analisis_keputusan(row):
    skor = row["tingkat_kerusakan_skor"]
    kategori = row["kategori"]

    if skor >= 80:
        prioritas = "PRIORITAS TINGGI (Darurat)"
        anggaran = "Rp 2.500.000"
    elif skor >= 60:
        prioritas = "PRIORITAS SEDANG (Terjadwal)"
        anggaran = "Rp 1.200.000"
    else:
        prioritas = "PRIORITAS RENDAH (Rutin)"
        anggaran = "Rp 500.000"
    if "Arborist" in kategori:
        tim = "Tim Arborist (Pemangkasan/Penebangan)"
    elif "Berm" in kategori:
        tim = "Tim Lanskap (Pemotong Rumput Kota)"
    else:
        tim = "Tim Teknisi Fasilitas Publik"

    return pd.Series([prioritas, tim, anggaran])

gdf[["status_prioritas", "tim_penanganan", "estimasi_anggaran"]] = gdf.apply(analisis_keputusan, axis=1)

gdf.to_file("hasil_analisis_aset.geojson", driver="GeoJSON")
gdf.drop(columns="geometry").to_csv("laporan_rekap_council.csv", index=False)

print("Analisis Spasial selesai! File 'hasil_analisis_aset.geojson' dan 'laporan_rekap_council.csv' berhasil dibuat.\n")

gdf[["id_laporan", "nama_aset", "jarak_posko_m", "status_prioritas", "tim_penanganan", "estimasi_anggaran"]]

Analisis Spasial selesai! File 'hasil_analisis_aset.geojson' dan 'laporan_rekap_council.csv' berhasil dibuat.



,id_laporan,nama_aset,jarak_posko_m,status_prioritas,tim_penanganan,estimasi_anggaran
0,AST-001,Pohon Pelindung Jl. WR Supratman (Dekat Gerban...,135.5,PRIORITAS TINGGI (Darurat),Tim Arborist (Pemangkasan/Penebangan),Rp 2.500.000
1,AST-002,Area Rumput (Berm) Simpang Kandang Limun,511.5,PRIORITAS RENDAH (Rutin),Tim Lanskap (Pemotong Rumput Kota),Rp 500.000
2,AST-003,Pohon Tua Depan Dekanat FT UNIB,183.9,PRIORITAS TINGGI (Darurat),Tim Arborist (Pemangkasan/Penebangan),Rp 2.500.000
3,AST-004,Lampu & Bangku Taman Danau Dendam Tak Sudah,4934.5,PRIORITAS SEDANG (Terjadwal),Tim Teknisi Fasilitas Publik,Rp 1.200.000
4,AST-005,Pohon Rawan Tumbang Jl. Kalimantan (Rawa Makmur),1123.9,PRIORITAS TINGGI (Darurat),Tim Arborist (Pemangkasan/Penebangan),Rp 2.500.000
5,AST-006,Taman Bermain Publik Pantai Panjang,6056.8,PRIORITAS RENDAH (Rutin),Tim Teknisi Fasilitas Publik,Rp 500.000
6,AST-007,Area Rumput (Berm) Rumah Lansia Jl. Indragiri,4774.3,PRIORITAS RENDAH (Rutin),Tim Lanskap (Pemotong Rumput Kota),Rp 500.000
7,AST-008,Pohon Miring Dekat RSUD M. Yunus,9131.4,PRIORITAS TINGGI (Darurat),Tim Arborist (Pemangkasan/Penebangan),Rp 2.500.000


In [6]:
import folium

peta = folium.Map(location=[-3.7880, 102.2850], zoom_start=13, tiles="CartoDB positron")

folium.TileLayer("OpenStreetMap", name="Peta Jalan (OpenStreetMap)").add_to(peta)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Citra Satelit (Esri World Imagery)"
).add_to(peta)

koordinat_posko = [-3.7585, 102.2725]
folium.Marker(
    location=koordinat_posko,
    popup=folium.Popup("<b>POSKO UTAMA CITYCARE</b><br>Pusat Komando & Alat Berat (Area UNIB)", max_width=250),
    tooltip="Posko Utama Tim Maintenance",
    icon=folium.Icon(color="blue", icon="home")
).add_to(peta)

warna_pin = {
    "PRIORITAS TINGGI (Darurat)": "red",
    "PRIORITAS SEDANG (Terjadwal)": "orange",
    "PRIORITAS RENDAH (Rutin)": "green"
}

for _, row in gdf.iterrows():
    warna = warna_pin[row["status_prioritas"]]
    lokasi_aset = [row["latitude"], row["longitude"]]

    if row["status_prioritas"] == "PRIORITAS TINGGI (Darurat)":
        folium.Circle(
            location=lokasi_aset,
            radius=150,
            color="crimson",
            weight=2,
            fill=True,
            fill_opacity=0.25,
            tooltip=f"Zona Bahaya Rawan Tumbang (Radius 150m) - {row['id_laporan']}"
        ).add_to(peta)

    folium.PolyLine(
        locations=[koordinat_posko, lokasi_aset],
        color=warna,
        weight=1.5,
        opacity=0.6,
        dash_array="5, 5"
    ).add_to(peta)

    isi_popup = f"""
    <div style="font-family: Arial, sans-serif; width: 250px; font-size: 12px;">
        <h4 style="margin: 0 0 6px 0; color: #2c3e50; border-bottom: 2px solid {warna}; padding-bottom: 4px;">
            {row['id_laporan']} | {row['kategori']}
        </h4>
        <b>Lokasi Aset:</b> {row['nama_aset']}<br>
        <b>Laporan Warga:</b> {row['kondisi_dilaporkan']}<br>
        <b>Skor Kerusakan:</b> {row['tingkat_kerusakan_skor']} / 100<br>
        <b>Jarak dari Posko:</b> {row['jarak_posko_m']} meter<br>
        <b>Tim Lapangan:</b> {row['tim_penanganan']}<br>
        <b>Estimasi Biaya:</b> {row['estimasi_anggaran']}<br>
        <div style="margin-top: 6px; padding: 4px; background-color: {warna}; color: white; text-align: center; font-weight: bold; border-radius: 4px;">
            {row['status_prioritas']}
        </div>
    </div>
    """

    folium.Marker(
        location=lokasi_aset,
        popup=folium.Popup(isi_popup, max_width=280),
        tooltip=f"Klik detail: {row['id_laporan']} - {row['nama_aset']}",
        icon=folium.Icon(color=warna, icon="info-sign")
    ).add_to(peta)

folium.LayerControl().add_to(peta)

peta.save("webmap_manajemen_aset.html")
print("Peta WebGIS Interaktif berhasil dibuat dan disimpan sebagai 'webmap_manajemen_aset.html'\n")

peta

Peta WebGIS Interaktif berhasil dibuat dan disimpan sebagai 'webmap_manajemen_aset.html'



/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
